# LoRA: Low-Rank Adaptation

> How much GPU memory does it take to fine-tune a 70B parameter model? About 140 GB for the model weights plus 280 GB for optimizer state -- at least 8 A100 GPUs. Most teams simply cannot assemble that kind of hardware.
>
> But what if updating fewer than 1% of the parameters could produce results close to full fine-tuning? In this section we start from the mathematics of low-rank matrix decomposition, implement a `LoraLinear` module, plug it into MiniGPT, and verify this claim hands-on.

The core assumption behind LoRA (Low-Rank Adaptation) is that although the weight matrices in an LLM are large -- say 4096 x 4096 -- the *change* needed during fine-tuning, $\Delta W$, is low-rank. It can be represented as the product of two small matrices.

Concretely, the original weight $W$ is frozen, and a side-path $AB$ is added alongside it ($A$ and $B$ are both small matrices). Only $A$ and $B$ are trained. After training, $AB$ is merged back into $W$, so at inference time the model is identical to a normally fine-tuned one -- with zero extra computational overhead.

This simple idea makes it possible to fine-tune a 70B model on a single RTX 4090.


## 1. The Cost of Full Fine-Tuning

Let us do the math first. During fine-tuning, GPU memory is consumed by several components:

| Component | Memory formula | LLaMA-7B | LLaMA-70B |
|-----------|---------------|----------|----------|
| Model weights (FP16) | `2 x N_bytes` | 14 GB | 140 GB |
| Optimizer state (Adam) | `12 x N_bytes` | 84 GB | 840 GB |
| Gradients | `2 x N_bytes` | 14 GB | 140 GB |
| Activations | depends on batch | ~8 GB | ~20 GB |
| **Total (full fine-tuning)** | | **~120 GB** | **~1140 GB** |
| **Total (LoRA, r=16)** | only LoRA params need optimizer state | **~15 GB** | **~145 GB** |

The core reason: full fine-tuning updates every parameter, so optimizer state and gradients must cover all of them. LoRA **does not update the original weights** -- they are frozen, and only the small side-path matrices are trained.

## 2. Low-Rank Weight Updates

A pretrained model has already learned general language patterns -- grammar, commonsense, reasoning. To adapt it to a new task, there is no need to overturn these patterns; we only need to nudge them along a few directions.

In mathematical terms: the pretrained weight $W$ is a $d \times d$ matrix (e.g., $d=4096$) with full rank $d$. But the update $\Delta W$ needed for a downstream task, while also $d \times d$ in size, has far less "information content" than $d \times d$ -- all columns can be expressed as linear combinations of a small number of basis vectors. In other words, $\Delta W$ is low-rank.

This means $\Delta W$ can be decomposed into the product of two small matrices:

```
delta_W (4096 x 4096) = A (4096 x r) x B (r x 4096)
Parameters: from 16M to r x (4096 + 4096) = r x 8192

When r=16: 16 x 8192 = 131,072 parameters
Compression ratio: 16M / 131K = ~128x
```

$r$ is the "rank" -- it determines how many basis vectors are used to represent $\Delta W$. A larger $r$ means more expressive power but also more trainable parameters. In practice $r=8$ or $r=16$ is usually sufficient.

**Key insight**: LoRA is not simply "training fewer parameters." It changes how the update is *represented*. Full fine-tuning directly learns every element of $\Delta W$; LoRA assumes $\Delta W$ is low-rank and only learns the two small matrices $A$ and $B$, using their product to approximate $\Delta W$.


## 3. LoRA Forward Pass

The core formula:

$$h = Wx + \frac{\alpha}{r} \cdot A B x$$

Where:
- $W$: original weight (frozen, not updated)
- $A$: small matrix of shape `(d_out, r)` (randomly initialized, trainable)
- $B$: small matrix of shape `(r, d_in)` (initialized to zero, trainable)
- $\alpha$: scaling factor, usually a small multiple of $r$
- $r$: rank, typically `r in {8, 16, 32, 64}`

**Why is $B$ initialized to zero?** This ensures that at the start of training $\Delta W = AB = 0$, so the model behaves exactly like the original. Fine-tuning introduces changes *gradually*, rather than disrupting behavior from the very first step.

Let us work through a tiny example by hand: d_in=4, d_out=4, r=2.

In [ ]:
import torch
# ============================================================
# Hand-compute LoRA: d_in=4, d_out=4, r=2
# ============================================================
torch.manual_seed(42)

d_in, d_out, r = 4, 4, 2
alpha = 2  # scaling factor

# Original weight (pretrained, frozen)
W = torch.randn(d_out, d_in)
print("Original weight W (4x4):")
print(W)
print(f"  Parameters: {W.numel()}")

# LoRA's two small matrices
A = torch.randn(d_out, r) * 0.02   # small random initialization
B = torch.zeros(r, d_in)            # <-- key: B initialized to 0!
print(f"\nA (d_out x r = 4x2):")
print(A)
print(f"B (r x d_in = 2x4), initialized to 0:")
print(B)
print(f"  A params: {A.numel()}, B params: {B.numel()}, total: {A.numel() + B.numel()}")
print(f"  Compression ratio: {W.numel() / (A.numel() + B.numel()):.1f}x")

# Input vector x
x = torch.tensor([1.0, 0.5, -0.5, -1.0])
print(f"\nInput x (4,): {x}")

# ============================================================
# Forward pass
# ============================================================
# Original output (without LoRA)
h_original = W @ x
print(f"\nOriginal output Wx: {h_original}")

# LoRA side-path
h_lora = B @ x     # Step 1: (r, d_in) @ (d_in,) -> (r,)    reduce dimension
h_lora = A @ h_lora  # Step 2: (d_out, r) @ (r,)  -> (d_out,) increase dimension
scaling = alpha / r
delta = h_lora * scaling
print(f"\nLoRA side-path computation:")
print(f"  Step 1 - Bx (r={r},): {h_lora}")
print(f"  Step 2 - A(Bx) (d_out={d_out},): {h_lora}")
print(f"  Scaling factor alpha/r: {scaling}")
print(f"  delta = (alpha/r) * A(Bx): {delta}")

# Final output: original + LoRA correction
h_final = h_original + delta
print(f"\nFinal output = Wx + (alpha/r)*ABx: {h_final}")

# Verify: when B=0, delta=0, output equals original
print(f"\nVerify B=0 => delta is all zeros: {(h_final == h_original).all().item()}")
print("(So at the start of training, model behavior is completely unchanged!)")


## 4. Implementing LoraLinear

Now let us turn the math into code. The core logic of `LoraLinear` is:

1. Internally hold a regular `nn.Linear` (the original weight $W$, frozen)
2. Create two additional small matrices $A$ and $B$ (trainable)
3. In forward: output = $Wx + (\alpha/r) \times A(Bx)$

The interface is identical to `nn.Linear` -- pass in `in_features` and `out_features`, plus a rank $r$. All LoRA complexity is encapsulated inside `forward`.

Here is the implementation:

In [ ]:
import torch.nn as nn
# ============================================================
# LoraLinear: wraps nn.Linear with a LoRA side-path
# ============================================================
import torch

class LoraLinear(nn.Module):
    """
    Equivalent to: y = Wx + (alpha/r) * A(Bx)
    - W: original weight (requires_grad=False, frozen)
    - A: (out_features, r) small matrix (trainable)
    - B: (r, in_features)  small matrix (trainable)
    """
    def __init__(self, linear: nn.Linear, r: int, alpha: int = None):
        super().__init__()
        # Keep reference to the original layer
        self.linear = linear
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.r = r
        self.alpha = alpha if alpha is not None else r

        # Freeze original weight
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # LoRA params: A random init, B zero init
        # A: (out_features, r)  B: (r, in_features)
        self.lora_A = nn.Parameter(torch.randn(self.out_features, r) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, self.in_features))
        self.scaling = self.alpha / self.r

    def forward(self, x):
        # Step 1: original output (frozen part, no gradient)
        original = self.linear(x)  # x @ W^T + bias
        # Step 2: LoRA side-path -> dropout stabilizes training, optional when r is small
        #   mathematically: x @ (lora_A @ lora_B)^T * scaling, i.e. (B^T @ A^T @ x^T)^T
        #   equivalent to:  (x @ lora_B^T) @ lora_A^T * scaling
        lora_out = (x @ self.lora_B.T) @ self.lora_A.T * self.scaling
        return original + lora_out

    @property
    def weight(self):
        """Merged equivalent weight: W + (alpha/r) * AB"""
        with torch.no_grad():
            delta_W = self.lora_A @ self.lora_B * self.scaling
            return self.linear.weight + delta_W

# Test
torch.manual_seed(42)
linear = nn.Linear(4, 4, bias=False)
linear.weight.data = torch.randn(4, 4)  # manually set weight

lora_layer = LoraLinear(linear, r=2, alpha=2)

print("=== LoraLinear parameter check ===")
print(f"  Original weight requires_grad: {lora_layer.linear.weight.requires_grad}")
print(f"  lora_A requires_grad: {lora_layer.lora_A.requires_grad}")
print(f"  lora_B requires_grad: {lora_layer.lora_B.requires_grad}")

# Count trainable parameters
total = sum(p.numel() for p in lora_layer.parameters())
trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
print(f"\n  Total params: {total}, trainable: {trainable} ({100*trainable/total:.1f}%)")

# Forward pass test
x = torch.tensor([[1.0, 0.5, -0.5, -1.0]])
y = lora_layer(x)
print(f"\n  Input x: {x.tolist()}")
print(f"  Output y: {y.tolist()}")


## 5. Validating LoRA

Before plugging LoRA into a real LLM, let us first validate its core assumption on a synthetic task: **can a low-rank matrix approximate the full fine-tuning update?**

We create a simple regression task: 30 data points, learning a `4 -> 4` linear mapping (ground truth defined by a random 4x4 matrix). We compare three approaches:
- **Full fine-tuning**: update all 16 parameters (full learning capacity)
- **LoRA (r=2)**: only update 2x4 + 4x2 = 16 parameters... wait, 16? Right, because 4x4=16. When the matrix itself is small (4x4), LoRA with r=2 has the same parameter count (A: 4x2=8, B: 2x4=8, total 16) as full fine-tuning (16), so r=2 should be able to learn the same solution. If r=1 (8 parameters), it should do worse -- exactly validating the principle that "higher rank means better approximation."
- **No training**: baseline

This experiment is small, but it directly validates whether LoRA's core assumption holds.

In [ ]:
import torch.nn.functional as F
# ============================================================
# Toy task: learn a linear mapping
# ============================================================
import torch
import torch.nn as nn

torch.manual_seed(42)

# Generate data: learn y = X @ W_target
d = 4
W_target = torch.tensor([
    [ 0.5, -0.3,  0.8, -0.2],
    [-0.4,  0.6, -0.1,  0.7],
    [ 0.3, -0.5, -0.6,  0.4],
    [-0.7,  0.2,  0.5, -0.8],
])

N = 30
X_train = torch.randn(N, d)
y_train = X_train @ W_target + 0.05 * torch.randn(N, d)  # add a little noise

print(f"Training data: X ({N}x{d}), y ({N}x{d})")
print(f"Target mapping W_target:\n{W_target}")

# ============================================================
# Approach 1: Full fine-tuning
# ============================================================
torch.manual_seed(100)
W_full = nn.Linear(d, d, bias=False)
W_full.weight.data = torch.randn(d, d)  # random init (simulating pretrained weights)

print(f"\n=== Approach 1: Full fine-tuning ===")
print(f"Initial weight:\n{W_full.weight.data}")
print(f"Trainable params: {sum(p.numel() for p in W_full.parameters())}")

opt_full = torch.optim.Adam(W_full.parameters(), lr=0.01)
losses_full = []
for epoch in range(200):
    opt_full.zero_grad()
    loss = F.mse_loss(W_full(X_train), y_train)
    loss.backward()
    opt_full.step()
    losses_full.append(loss.item())

print(f"  Final loss: {losses_full[-1]:.6f}")

# ============================================================
# Approach 2: LoRA (r=2)
# ============================================================
torch.manual_seed(100)  # same initial weight
W_base = nn.Linear(d, d, bias=False)
W_base.weight.data = torch.randn(d, d)  # same init as full fine-tuning

# Wrap with LoRA
lora_model = LoraLinear(W_base, r=2, alpha=4)

print(f"\n=== Approach 2: LoRA (r=2) ===")
print(f"Original weight (frozen):\n{lora_model.linear.weight.data}")
print(f"Trainable params: {sum(p.numel() for p in lora_model.parameters() if p.requires_grad)}")

opt_lora = torch.optim.Adam(lora_model.parameters(), lr=0.01)
losses_lora = []
for epoch in range(200):
    opt_lora.zero_grad()
    loss = F.mse_loss(lora_model(X_train), y_train)
    loss.backward()
    opt_lora.step()
    losses_lora.append(loss.item())

print(f"  Final loss: {losses_lora[-1]:.6f}")

# ============================================================
# Comparison
# ============================================================
print(f"\n=== Comparison ===")
print(f"  Full fine-tuning loss: {losses_full[-1]:.6f} (updates {d*d} params)")
print(f"  LoRA r=2         loss: {losses_lora[-1]:.6f} (updates {2*d*2} params)")

# Check if LoRA's learned weight is close to the target
with torch.no_grad():
    learned_W = lora_model.weight  # merged equivalent weight
    print(f"\n  LoRA learned equivalent weight:\n{learned_W}")
    print(f"  Target weight:\n{W_target}")
    print(f"  Equivalent weight vs target MSE: {F.mse_loss(learned_W, W_target):.6f}")

    full_W = W_full.weight.data
    print(f"  Full fine-tuning weight vs target MSE: {F.mse_loss(full_W, W_target):.6f}")


## 6. Plugging into MiniGPT: Adding LoRA to Attention

Recall MiniGPT's attention layer:
```
Q = W_Q @ X
K = W_K @ X
V = W_V @ X
O = W_O @ concat(heads)
```

We add LoRA to the Q and V projection layers (the most common practice), and freeze everything else.

In [ ]:
import math
# ============================================================
# Reuse MiniGPT from earlier parts
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=128):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer('pos_encoding', get_sinusoidal_encoding(max_len, d_model))
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        B, S = x.shape
        x_emb = self.token_embedding(x) + self.pos_encoding[:S, :]
        for block in self.blocks:
            x_emb = block(x_emb, mask)
        x_emb = self.norm_final(x_emb)
        return self.lm_head(x_emb)

# Causal mask
def causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

print("MiniGPT loaded")


In [ ]:
# ============================================================
# Add LoRA to MiniGPT's Attention layers
# ============================================================
def apply_lora_to_attention(model, r=16, alpha=32):
    """Add LoRA to W_Q and W_V in all attention layers"""
    lora_params = 0
    for block in model.blocks:
        attn = block.attention
        # Replace W_Q
        attn.W_Q = LoraLinear(attn.W_Q, r=r, alpha=alpha)
        # Replace W_V
        attn.W_V = LoraLinear(attn.W_V, r=r, alpha=alpha)
        # Count
        lora_params += attn.W_Q.lora_A.numel() + attn.W_Q.lora_B.numel()
        lora_params += attn.W_V.lora_A.numel() + attn.W_V.lora_B.numel()
    return lora_params

VOCAB_SIZE = 50
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

# Count original trainable params
total_params = sum(p.numel() for p in model.parameters())
print(f"Original MiniGPT total params: {total_params:,}")

# Apply LoRA
lora_params = apply_lora_to_attention(model, r=8, alpha=16)

# Recount
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nAfter adding LoRA:")
print(f"  Frozen params: {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")
print(f"  Trainable params (LoRA): {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"  Using only {100*trainable_params/total_params:.2f}% of params!")

# Verify: original weights are indeed frozen
for name, param in model.named_parameters():
    if 'lora' not in name and 'lm_head' not in name and 'token_embedding' not in name:
        if param.requires_grad:
            print(f"  WARNING: {name} should be frozen but requires_grad=True")
print(f"\nOriginal transformer weights are all frozen")


## 7. Training Demo: SFT with LoRA

The toy experiment proved that LoRA works in principle, but the real test is: plug LoRA into an actual LLM, train with real conversational data for SFT, and see if the loss decreases steadily.

Here we use MiniGPT + synthetic conversational data to compare two approaches:
- **Full fine-tuning**: update all of MiniGPT's parameters
- **LoRA**: only add LoRA side-paths to the Q and V projection matrices in Attention (the most common configuration, since Attention is where the model learns "how to read the input," and fine-tuning there gives the most benefit)

If LoRA's loss curve is close to full fine-tuning, it shows that LoRA is equally effective in a real scenario -- using fewer than 1% trainable parameters.

In [ ]:
# ============================================================
# Prepare SFT training data
# ============================================================
# Use a simple vocabulary to simulate conversations
# Vocabulary (50 tokens) -- only a subset is used
word_to_id = {
    "BOS": 0, "EOS": 1,
    "hello": 2, "thanks": 3, "goodbye": 4, "today": 5, "weather": 6, "nice": 7,
    "you": 8, "are": 9, "who": 10, "I": 11, "assistant": 12, "AI": 13,
    "what": 14, "why": 15, "how": 16, "do": 17, "can": 18,
    "good": 19, "bad": 20, "like": 21, "dislike": 22, "know": 23, "unk": 24,
    "math": 25, "english": 26, "code": 27, "write": 28, "program": 29,
    "today": 5, "week": 30, "one": 31, "two": 32, "three": 33,
}

# Clean up duplicates
word_to_id = {
    "BOS": 0, "EOS": 1, "PAD": 0,
    "hello": 2, "thanks": 3, "goodbye": 4, "today": 5, "weather": 6, "nice": 7,
    "you": 8, "are": 9, "who": 10, "I": 11, "assistant": 12, "AI": 13,
    "what": 14, "why": 15, "how": 16, "do": 17, "can": 18,
    "good": 19, "bad": 20, "like": 21, "dislike": 22, "know": 23, "unk": 24,
    "math": 25, "english": 26, "code": 27, "write": 28, "program": 29,
    "week": 30, "one": 31, "two": 32, "three": 33, "four": 34,
    "five": 35, "six": 36, "day": 37, "q": 38, "of": 39,
    "very": 40, "not": 41, "yes": 42, "no": 43, "um": 44,
    "ah": 45, "ok": 46, "will": 47, "did": 48, "at": 49,
}

# Generate training data: user question + assistant answer
conversations = [
    ("hello", "hello what can I do for you"),
    ("today weather nice", "yes today weather very good for going out"),
    ("you are who", "I am an AI assistant can answer your question"),
    ("you can write code q", "yes I can write code especially program"),
    ("thanks", "you are welcome what else do you want to know"),
    ("goodbye", "goodbye have a nice day"),
]

def encode_text(text):
    """Simple word-level encoding"""
    ids = []
    for word in text.split():
        if word in word_to_id:
            ids.append(word_to_id[word])
        else:
            ids.append(0)  # unknown word -> PAD/BOS
    return ids

print("Training data examples:")
for q, a in conversations[:3]:
    print(f"  User: {q}  ->  Assistant: {a}")
    q_ids = encode_text(q)
    a_ids = encode_text(a)
    print(f"    IDs: User {q_ids}, Assistant {a_ids}")


In [ ]:
# ============================================================
# LoRA SFT training loop
# ============================================================
import torch
import torch.nn.functional as F

VOCAB_SIZE = 50
SEQ_LEN = 24
torch.manual_seed(42)

# Create model and apply LoRA
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2, max_len=SEQ_LEN)
apply_lora_to_attention(model, r=8, alpha=16)

# Only optimize trainable params (LoRA's A and B + lm_head + embedding)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.001
)

print(f"Parameters managed by optimizer: {sum(p.numel() for p in optimizer.param_groups[0]['params']):,}")

# ============================================================
# Training
# ============================================================
NUM_EPOCHS = 50
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0

    for user_text, assistant_text in conversations:
        # Build full input sequence: [BOS] + user + assistant + [EOS]
        user_ids = encode_text(user_text)
        asst_ids = encode_text(assistant_text)
        full_seq = [word_to_id["BOS"]] + user_ids + asst_ids + [word_to_id["EOS"]]

        # Pad to fixed length
        if len(full_seq) > SEQ_LEN:
            full_seq = full_seq[:SEQ_LEN]
        input_ids = torch.tensor([full_seq + [0] * (SEQ_LEN - len(full_seq))])

        # Forward pass
        mask = causal_mask(SEQ_LEN)
        logits = model(input_ids, mask)  # (1, S, V)

        # Labels: shift right by one position
        labels = torch.cat([input_ids[:, 1:], torch.zeros(1, 1, dtype=torch.long)], dim=1)

        # Loss: only compute on non-PAD positions
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            labels.view(-1),
            ignore_index=0  # PAD
        )

        epoch_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = epoch_loss / len(conversations)
    losses.append(avg_loss)

    if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f}")

print(f"\nInitial Loss: {losses[0]:.4f} -> Final Loss: {losses[-1]:.4f}")
print(f"Loss decrease: {losses[0] - losses[-1]:.4f}")


In [ ]:
# ============================================================
# Visualize loss decrease
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(losses, 'b-', linewidth=1.5)
plt.axhline(y=losses[0], color='gray', linestyle='--', alpha=0.5, label=f'Initial: {losses[0]:.2f}')
plt.axhline(y=losses[-1], color='red', linestyle='--', alpha=0.5, label=f'Final: {losses[-1]:.2f}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LoRA fine-tuning MiniGPT -- loss curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nConclusion: updating fewer than 1% of parameters, the loss still decreases significantly!")
print("This is the value of LoRA -- with very few trainable parameters, on suitable tasks and ranks, approaching full fine-tuning performance; whether it actually approaches must be verified with a validation set and task evaluation.")


## 8. Merging Weights for Inference

LoRA has a very practical design: after training, the side-path matrices $AB$ can be "absorbed" back into the original weight.

The procedure is to compute `W_new = W + (alpha/r) * A @ B`, then store it as the new model weight. During inference, forward is simply `x @ W_new` -- no extra LoRA side-path computation needed.

This means:
- **Inference speed unchanged**: if LoRA weights are merged into a single model ahead of time, the forward computation is identical to a plain Linear; if adapters are swapped dynamically at runtime, there is still additional management or computation overhead
- **No extra memory**: no need to keep $A$ and $B$ in GPU memory
- **Deployment compatible**: the merged model can be used directly with existing inference frameworks (vLLM, TensorRT-LLM, etc.) without any modifications

This is why LoRA is so popular in industry -- cheap to train, unchanged inference, low deployment cost -- but you still need to handle multi-adapter management, merged-weight storage, and regression testing.

In [ ]:
# ============================================================
# Merge LoRA weights into original weights
# ============================================================
import torch.nn as nn

def merge_lora(model):
    """Merge LoRA weights into original weights; after merge, no LoRA side-path is used"""
    for block in model.blocks:
        attn = block.attention
        for name in ['W_Q', 'W_V']:
            layer = getattr(attn, name)
            if isinstance(layer, LoraLinear):
                # Compute merged weight
                merged_weight = layer.linear.weight.data + layer.lora_A.data @ layer.lora_B.data * layer.scaling
                # Replace with plain nn.Linear
                new_linear = nn.Linear(layer.in_features, layer.out_features, bias=False)
                new_linear.weight.data = merged_weight
                setattr(attn, name, new_linear)

# Before/after comparison
trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params before merge: {trainable_before}")

merge_lora(model)

trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after merge: {trainable_after}")
print(f"\nAfter merging, inference speed is exactly the same as a regular model -- no LoRA overhead!")


## 9. Practical Tips Summary

| Question | Answer |
|----------|--------|
| Which layers to add LoRA to? | Early practice often adds only Q/V; modern practice frequently adds Q/K/V/O, FFN, or lm_head -- must be verified empirically |
| What rank r to use? | r=8/16 is a common starting point, not guaranteed to be sufficient; higher rank means more expressive power, but also more parameters and memory |
| How to set alpha? | Usually 2-4x r. alpha/r is the actual scaling factor controlling the strength of LoRA updates |
| How many LoRAs per base model? | Theoretically unlimited. Each LoRA is only a few MB, can be swapped at will, enabling "one base + multiple task adapters" |
| Performance gap vs full fine-tuning? | Depends on the task, data volume, rank, target modules, and training setup; cannot be summarized as a fixed percentage |
| What is QLoRA? | LoRA + 4-bit quantization. NF4 works well for approximately normally-distributed weights; the memory saving ratio depends on the model, sequence length, batch, and optimizer |
| How to deploy after training? | You can merge A x B into the original weights; after merge, single-model inference has no LoRA side-path overhead, but dynamic adapter serving still has additional management cost |

**One-sentence summary**: LoRA's core assumption is that "weight updates for downstream tasks are low-rank." Using this assumption, it compresses billions of trainable parameters down to hundreds of thousands. The number of trainable parameters can be reduced a lot, but whether performance approaches full fine-tuning must be verified with task evaluation.

## 10. QLoRA

LoRA has already reduced trainable parameters from billions to hundreds of thousands, but the base model still needs to be loaded into GPU memory at full precision.
A 7B parameter model requires 14 GB for weights alone (FP16), not counting optimizer state and activations.

QLoRA asks: **can the base model also be compressed?**

The answer is yes. QLoRA uses three key techniques to reduce memory from ~15 GB to ~6 GB (for a 7B model):

- `NF4` (4-bit NormalFloat): a 4-bit quantization format specifically designed for normally-distributed weights,
  information-theoretically optimal
- `Double Quantization`: quantize the quantization constants themselves, further saving memory
- `Paged Optimizers`: use unified memory management to avoid memory spikes from optimizer state

Memory comparison for three approaches on a 7B model:

| Approach | Base model precision | Trainable params | Total memory (estimated) |
|:---------|:---------------------|:-----------------|:------------------------|
| Full fine-tuning | FP16 | 7B | ~120 GB |
| LoRA | FP16 | ~4M | ~15 GB |
| QLoRA | NF4 (4-bit) | ~4M | ~6 GB |

In [ ]:
# ============================================================
# QLoRA memory comparison: full fine-tuning vs LoRA vs QLoRA (7B model)
# ============================================================

# 7B model parameter count
params_7B = 7e9

# LoRA trainable params: assume r=16, applied to Q/V projections
# Each layer has 2 LoRA groups, each LoRA param count = d_out * r + r * d_in = 2 * d^2 * r / d
# Simplified: 7B model has ~32 layers, d=4096, r=16
num_layers = 32
d_model = 4096
lora_r = 16
lora_params_per_layer = 2 * (d_model * lora_r + lora_r * d_model)  # Q and V each have one LoRA
total_lora_params = num_layers * lora_params_per_layer

print(f"=== 7B model memory estimate ===")
print(f"Total params: {params_7B/1e9:.0f}B")
print(f"LoRA trainable params: {total_lora_params:,} ({total_lora_params/params_7B*100:.2f}%)")
print()

# 1) Full fine-tuning
model_fp16 = params_7B * 2 / 1e9       # 2 bytes per param
optimizer_adam = params_7B * 12 / 1e9   # Adam: 12 bytes per param (momentum + variance + master weights)
gradients = params_7B * 2 / 1e9          # 2 bytes per param
activations = 8  # estimate
full_total = model_fp16 + optimizer_adam + gradients + activations

print(f"--- Full fine-tuning ---")
print(f"  Model weights (FP16):       {model_fp16:.0f} GB")
print(f"  Optimizer state (Adam):      {optimizer_adam:.0f} GB")
print(f"  Gradients:                   {gradients:.0f} GB")
print(f"  Activations:                 ~{activations} GB")
print(f"  Total:                       ~{full_total:.0f} GB")
print()

# 2) LoRA
# Base model FP16 (frozen), optimizer only handles LoRA params
lora_model = model_fp16
lora_optim = total_lora_params * 12 / 1e9
lora_grad = total_lora_params * 2 / 1e9
lora_total = lora_model + lora_optim + lora_grad + activations

print(f"--- LoRA (r={lora_r}) ---")
print(f"  Base model (FP16, frozen):   {lora_model:.0f} GB")
print(f"  Optimizer state (LoRA only): {lora_optim:.2f} GB")
print(f"  Gradients (LoRA only):       {lora_grad:.3f} GB")
print(f"  Activations:                 ~{activations} GB")
print(f"  Total:                       ~{lora_total:.0f} GB")
print()

# 3) QLoRA
# Base model uses NF4 quantization (4 bits = 0.5 byte per param)
# Optimizer still only handles LoRA params
qlora_model = params_7B * 0.5 / 1e9  # 4-bit = 0.5 byte per param
qlora_optim = lora_optim
qlora_grad = lora_grad
qlora_double_quant = 0.4  # Double Quantization saves ~0.4 GB of quantization constants
qlora_total = qlora_model + qlora_optim + qlora_grad + activations + qlora_double_quant

print(f"--- QLoRA (r={lora_r}, NF4) ---")
print(f"  Base model (NF4, frozen):    {qlora_model:.1f} GB")
print(f"  Double Quantization:         +{qlora_double_quant} GB (quantization constants)")
print(f"  Optimizer state (LoRA only): {qlora_optim:.2f} GB")
print(f"  Gradients (LoRA only):       {qlora_grad:.3f} GB")
print(f"  Activations:                 ~{activations} GB")
print(f"  Total:                       ~{qlora_total:.0f} GB")
print()

# Summary comparison
print(f"=== Key comparison ===")
print(f"  Full fine-tuning -> LoRA:    memory from ~{full_total:.0f} GB to ~{lora_total:.0f} GB ({full_total/lora_total:.1f}x)")
print(f"  LoRA -> QLoRA:               memory from ~{lora_total:.0f} GB to ~{qlora_total:.0f} GB ({lora_total/qlora_total:.1f}x)")
print(f"  Full fine-tuning -> QLoRA:   memory from ~{full_total:.0f} GB to ~{qlora_total:.0f} GB ({full_total/qlora_total:.1f}x)")
print()
print("Key observation: QLoRA saves approximately 60% more memory than LoRA,")
print("  primarily from compressing the base model from FP16 (2 bytes) to NF4 (0.5 byte).")


In [ ]:
# ============================================================
# NF4 quantization demo: from FP16 to 4-bit and back
# ============================================================

import numpy as np

# Core idea of NF4:
# Neural network weights are approximately normally distributed N(0, sigma^2).
# NF4 divides this normal distribution into 16 equal-probability intervals (2^4 = 16 quantization levels),
# each containing the same proportion of weight values, minimizing information loss.

# 16 NF4 quantization levels (from the QLoRA paper)
nf4_levels = np.array([
    -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1880, -0.0950,  0.0,
     0.0594,  0.1524,  0.2500,  0.3536,  0.4714,  0.6107,  0.7906,  1.0
])

print("=== NF4 quantization levels (16) ===")
print(f"  Negative half: {nf4_levels[:8].tolist()}")
print(f"  Positive half: {nf4_levels[8:].tolist()}")
print(f"  Note: levels are denser near 0, sparser far from 0")
print(f"  This matches the normal distribution shape -- most weights cluster near 0")
print()

# ============================================================
# Hand-compute: quantize a specific weight value
# ============================================================

# Assume weight scaling factor (absmax) = 0.8
# During quantization, first normalize weights to [-1, 1], then find nearest NF4 level
absmax = 0.8

# 5 simulated FP16 weight values
weights_fp16 = np.array([0.12, -0.35, 0.67, -0.02, 0.51])

print(f"=== Hand-compute NF4 quantization ===")
print(f"Scaling factor absmax: {absmax}")
print(f"Original FP16 weights: {weights_fp16.tolist()}")
print()

# Step 1: normalize to [-1, 1]
weights_norm = weights_fp16 / absmax
print(f"Step 1 -- normalize (w / absmax): {[f'{v:.4f}' for v in weights_norm]}")

# Step 2: find nearest NF4 level (this is quantization)
quant_indices = []
quant_values = []
for w in weights_norm:
    # Find nearest neighbor
    idx = np.argmin(np.abs(nf4_levels - w))
    quant_indices.append(idx)
    quant_values.append(nf4_levels[idx])

quant_indices = np.array(quant_indices)
quant_values = np.array(quant_values)

print(f"Step 2 -- quantize to NF4 level indices: {quant_indices.tolist()}")
print(f"         Corresponding NF4 values:         {[f'{v:.4f}' for v in quant_values]}")
print(f"  (each index needs only 4 bits to store)")
print()

# Step 3: dequantize = NF4 value x absmax
weights_dequant = quant_values * absmax
print(f"Step 3 -- dequantize (NF4 value x absmax): {[f'{v:.4f}' for v in weights_dequant]}")
print()

# Quantization error
error = weights_fp16 - weights_dequant
print(f"=== Quantization error ===")
print(f"Original:      {[f'{v:7.4f}' for v in weights_fp16]}")
print(f"Dequantized:   {[f'{v:7.4f}' for v in weights_dequant]}")
print(f"Error:         {[f'{v:7.4f}' for v in error]}")
print(f"Max error:     {np.max(np.abs(error)):.4f}")
print(f"Relative error: {np.max(np.abs(error)) / np.max(np.abs(weights_fp16)) * 100:.1f}%")
print()
print("Key observation: 4-bit quantization has a max error of ~6%, but the paper found")
print("  that in LoRA fine-tuning scenarios, this error has negligible impact on final performance.")
print(f"\nStorage savings: FP16 uses 16 bits/param -> NF4 uses 4 bits/param = 4x compression")


QLoRA's complete training pipeline can be summarized in these steps:

1. When **loading the base model**, weights are quantized from FP16 to NF4 format, reducing memory to 1/4
2. LoRA adapters are added, and adapter weights remain in `bf16` precision (for normal training)
3. During forward pass, NF4 weights are first dequantized back to FP16 before participating in computation; gradients flow only to LoRA parameters
4. After training, LoRA weights are merged, and the model can optionally be restored to higher precision for deployment

Essentially, QLoRA is the combination of **quantization + LoRA**: using NF4 to compress the base model's memory,
and LoRA to maintain training effectiveness. Neither changes the model architecture, nor adds computation during inference.

## 11. Model Merging

In the previous section we learned that LoRA weights can be merged back into the base model. The open-source community has pushed this idea further -- **merging multiple different fine-tuned models into one**, hoping the merged model simultaneously possesses multiple capabilities.

This is common in practice: someone fine-tunes a code model, someone fine-tunes a math model, someone fine-tunes a Chinese conversational model. Merging them together could produce an "all-around" model.

### Mainstream Merging Methods

The simplest merging approach is direct weight averaging, but results are often poor -- weight directions from different models may cancel each other out. Better methods include:

| Method | Core idea | Characteristics |
|:-------|:----------|:---------------|
| **SLERP** | Spherical linear interpolation, preserves weight directions | Best for merging two models |
| **TIES** | Identifies most important parameter updates, discards conflicts | Suitable for merging multiple models |
| **DARE** | Randomly drops fine-tuned parameters, then averages | Simple and efficient |
| **Task Arithmetic** | Uses "task vectors" (fine-tuned - base) for addition/subtraction | Intuitive, supports task addition/removal |

**Task Arithmetic** intuition:
```
task_vector = fine-tuned weights - base weights

merged model = base weights + alpha1 * task_vector1 + alpha2 * task_vector2 + ...
```
Each task vector captures "in which direction weights need to change for a particular capability." Overlaying them onto the base is like adding multiple capabilities simultaneously.

In [ ]:
# Demo: Task Arithmetic and SLERP
import numpy as np

np.random.seed(42)

# Simulate base model and two fine-tuned model weights
d = 8  # weight dimension (in reality millions of dimensions)
W_base = np.random.randn(d).astype(np.float32)

# Fine-tuned model 1: adjusted toward "code"
W_code = W_base + np.array([0.5, -0.3, 0.8, -0.1, 0.2, -0.4, 0.6, -0.2])
# Fine-tuned model 2: adjusted toward "math"
W_math = W_base + np.array([-0.2, 0.6, 0.1, 0.7, -0.3, 0.5, -0.1, 0.4])

# === Task Arithmetic ===
task_code = W_code - W_base  # code task vector
task_math = W_math - W_base  # math task vector
W_merged_arith = W_base + 0.5 * task_code + 0.5 * task_math

print("=== Task Arithmetic ===")
print(f"Base weights:      {np.round(W_base, 2)}")
print(f"Code task vector:  {np.round(task_code, 2)}")
print(f"Math task vector:  {np.round(task_math, 2)}")
print(f"Merged weights:    {np.round(W_merged_arith, 2)}")
print()

# === SLERP ===
def slerp(v1, v2, t=0.5):
    """Spherical linear interpolation"""
    v1_n = v1 / np.linalg.norm(v1)
    v2_n = v2 / np.linalg.norm(v2)
    dot = np.clip(np.dot(v1_n, v2_n), -1.0, 1.0)
    omega = np.arccos(dot)
    if omega < 1e-6:
        return (1 - t) * v1 + t * v2
    return (np.sin((1 - t) * omega) / np.sin(omega)) * v1 + (np.sin(t * omega) / np.sin(omega)) * v2

W_merged_slerp = slerp(W_code, W_math, t=0.5)
print("=== SLERP ===")
print(f"Code model weights:  {np.round(W_code, 2)}")
print(f"Math model weights:  {np.round(W_math, 2)}")
print(f"SLERP merge:         {np.round(W_merged_slerp, 2)}")
print(f"Simple average:      {np.round((W_code + W_math) / 2, 2)}")
print()
print("Key observation: SLERP preserves directional information from both weights, unlike simple averaging which flattens differences")


### mergekit: The Community Standard Merging Tool

In practice, model merging is done through **mergekit**, currently the most popular open-source merging tool. A typical configuration file looks like:

```yaml
# mergekit config example
models:
  - model: Qwen/Qwen2.5-7B          # base
    parameters:
      weight: 1.0
  - model: some-user/code-7B          # code fine-tune
    parameters:
      weight: 0.5
  - model: some-user/math-7B          # math fine-tune
    parameters:
      weight: 0.5
merge_method: ties
base_model: Qwen/Qwen2.5-7B
dtype: bfloat16
```

Run command: `mergekit-yaml config.yaml ./merged-output`

Note that model merging is not a silver bullet -- if two fine-tuning directions conflict (e.g., one specializes in Japanese, another in code style), the merged model may be worse at both. Merging works best when directions are complementary.

## Summary

- LoRA's core idea is decomposing the weight update into the product of two low-rank matrices A and B
- A is randomly initialized, B is initialized to zero, ensuring model behavior is unchanged at the start of training
- Only LoRA parameters are trained (~0.1% of total parameters), all other weights are frozen
- After training, A x B can be merged into the original weight; after merge, single-model inference has no LoRA side-path overhead
- QLoRA uses NF4 quantization on top of LoRA to compress the base model, reducing memory from ~15 GB to ~6 GB
- NF4 quantization levels are denser near 0, matching the normal distribution of weights, minimizing information loss
- QLoRA = NF4 quantization (compress base model) + LoRA (train only a few parameters)
- Model merging is a technique for fusing weights from multiple fine-tuned models into one (Task Arithmetic, SLERP, TIES, DARE)
- mergekit is a commonly used open-source model merging tool that defines merging strategies through a YAML config file

## Exercises

> You can ask AI to help explain ideas or break down steps, but it is not recommended to let AI "do the whole problem" for you.

**Exercise 1: LoRA parameter count**

LoRA decomposes a $d_{out} \times d_{in}$ linear layer into $A$ ($r \times d_{in}$) and $B$ ($d_{out} \times r$). Assume the original linear layer is $4096 \times 4096$ and LoRA rank $r = 16$. Compute:
1. The parameter count of the original linear layer
2. The new parameters added by LoRA (A + B)
3. The ratio of LoRA parameters to the original parameter count

Hint: original params = $d_{out} \times d_{in}$, LoRA params = $r \times d_{in} + d_{out} \times r$.

In [ ]:
# Exercise 1: LoRA parameter count
d_out, d_in = 4096, 4096
r = 16

# TODO: compute original linear layer parameter count
original_params = None  # compute here

# TODO: compute LoRA new parameters (A + B)
lora_params = None  # compute here

# TODO: compute the ratio
ratio = None  # lora_params / original_params

assert original_params is not None, "Please compute original_params first"
assert lora_params is not None, "Please compute lora_params first"
assert ratio is not None, "Please compute ratio first"

expected_orig = d_out * d_in
expected_lora = r * d_in + d_out * r
assert original_params == expected_orig, f"Original params should be {expected_orig}"
assert lora_params == expected_lora, f"LoRA params should be {expected_lora}"

print(f"✅ Exercise 1 passed:")
print(f"   Original params: {original_params:,}")
print(f"   LoRA params:     {lora_params:,}")
print(f"   Ratio:           {ratio:.4%}")
print("   LoRA only needs to train under 0.8% of the parameters, drastically reducing memory and compute.")


**Exercise 2: LoRA weight merging**

After LoRA training, you can merge $\Delta W = A \times B$ back into the original weight $W$ to get $W' = W + \alpha/r \cdot AB$. After merging, there is no extra overhead during inference.

Given a simplified 2x2 example:
- $W = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}$
- $A = \begin{pmatrix} 0.1 & 0.2 \end{pmatrix}$ ($1 \times 2$)
- $B = \begin{pmatrix} 0.3 \\ -0.1 \end{pmatrix}$ ($2 \times 1$, initially zero)
- $\alpha/r = 1$

Assume that after training, $B$ is updated to $\begin{pmatrix} 0.3 \\ -0.1 \end{pmatrix}$. Compute the merged weight $W'$.

Hint: $\Delta W = B \times A$ (outer product), $W' = W + \Delta W$. Pay attention to the dimensions of A and B.

In [ ]:
# Exercise 2: LoRA weight merging
import torch

W = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
A = torch.tensor([[0.1, 0.2]])       # 1 x 2 (r=1, d_in=2)
B = torch.tensor([[0.3], [-0.1]])     # 2 x 1 (d_out=2, r=1)
scaling = 1.0  # alpha/r = 1

# TODO: compute delta_W = B @ A (outer product, mind the dimensions)
delta_W = None  # compute here

# TODO: compute the merged weight
W_merged = None  # W + scaling * delta_W

assert delta_W is not None, "Please compute delta_W first"
assert W_merged is not None, "Please compute W_merged first"

expected_delta = B @ A  # [2,1] @ [1,2] = [2,2]
expected_merged = W + scaling * expected_delta
assert torch.allclose(delta_W, expected_delta, atol=0.001), "delta_W is incorrect"
assert torch.allclose(W_merged, expected_merged, atol=0.001), "W_merged is incorrect"

print(f"✅ Exercise 2 passed:")
print(f"   delta_W = B @ A = {delta_W.tolist()}")
print(f"   W' = W + delta_W = {W_merged.tolist()}")
print("   The merged weight has the same shape as the original, with zero inference overhead.")


**Exercise 3: QLoRA memory savings**

QLoRA uses NF4 (4-bit) quantization on top of LoRA to compress the base model, while LoRA parameters are still trained in FP16. Assume a 7B model, LoRA rank $r=16$, applied to the Q and V projection layers (each a $4096 \times 4096$ linear layer). Compute:
1. The memory of the base model after NF4 quantization (4 bits per param)
2. The memory of the LoRA parameters (FP16, 2 bytes per param)
3. The total QLoRA training memory (model weights + LoRA params only, excluding optimizer)

Hint: base 7B x 0.5 bytes + LoRA param count x 2 bytes.

In [ ]:
# Exercise 3: QLoRA memory estimate
base_params = 7e9  # 7B
num_layers = 2     # Q and V
d = 4096
r = 16

# TODO: compute total LoRA parameter count (2*r*d per layer)
lora_params = None  # compute here

# TODO: base model NF4: 4 bits = 0.5 bytes per param
base_memory_gb = None  # compute here

# TODO: LoRA FP16: 2 bytes per param
lora_memory_gb = None  # compute here

# TODO: QLoRA total memory
total_memory_gb = None  # compute here

assert lora_params is not None, "Please compute lora_params first"
assert base_memory_gb is not None, "Please compute base_memory_gb first"
assert total_memory_gb is not None, "Please compute total_memory_gb first"

expected_lora = num_layers * 2 * r * d
expected_base = base_params * 0.5 / 1e9
expected_lora_mem = expected_lora * 2 / 1e9
expected_total = expected_base + expected_lora_mem
assert lora_params == expected_lora, f"LoRA params should be {expected_lora}"
assert abs(base_memory_gb - expected_base) < 0.1, f"Base memory should be {expected_base:.1f} GB"
assert abs(total_memory_gb - expected_total) < 0.1, f"Total memory should be {expected_total:.1f} GB"

print(f"✅ Exercise 3 passed:")
print(f"   Base NF4:   {base_memory_gb:.1f} GB")
print(f"   LoRA FP16:  {lora_memory_gb:.2f} GB")
print(f"   QLoRA total: {total_memory_gb:.1f} GB")
print("   Compared to ~120 GB for full FP16 training, QLoRA compresses memory to the ~4 GB level.")
